In [34]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

In [35]:
df = pd.read_csv("data/airbnb_listings_stockholm_detailed.csv")
print(df.shape)

(4955, 79)


In [36]:
df = df[[
    "price",
    "accommodates",
    "bedrooms",
    "bathrooms",
    "property_type",
    "room_type",
    "neighbourhood_cleansed",
    "review_scores_rating",
    "number_of_reviews",
    "availability_365",
    "minimum_nights"
]].copy()

print(df.shape)
df.head()

(4955, 11)


,price,accommodates,bedrooms,bathrooms,property_type,room_type,neighbourhood_cleansed,review_scores_rating,number_of_reviews,availability_365,minimum_nights
0,$917.00,2,1.0,1.0,Private room in rental unit,Private room,Södermalms,4.86,454,169,2
1,$450.00,1,1.0,1.0,Private room in rental unit,Private room,Kungsholmens,4.69,66,296,2
2,"$1,073.00",2,1.0,1.0,Entire rental unit,Entire home/apt,Norrmalms,4.79,110,79,1
3,$779.00,1,1.0,1.0,Private room in rental unit,Private room,Södermalms,4.90,444,168,2
4,"$2,352.00",4,3.0,2.0,Entire rental unit,Entire home/apt,Södermalms,5.00,4,271,6


In [37]:
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

df["price"].head()

0     917.0
1     450.0
2    1073.0
3     779.0
4    2352.0
Name: price, dtype: float64

In [38]:
df = df[df["price"] < 25000].copy()
df["price"].describe()

count     3185.000000
mean      1568.662166
std       1310.943528
min        120.000000
25%        753.000000
50%       1200.000000
75%       2000.000000
max      17872.000000
Name: price, dtype: float64

In [39]:
df = df[~(
    ((df["room_type"] == "Private room") & (df["accommodates"] > 6)) |
    ((df["room_type"] == "Shared room") & (df["accommodates"] > 8))
)].copy()

df["accommodates"].describe()

count    3176.000000
mean        3.487406
std         2.037406
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max        16.000000
Name: accommodates, dtype: float64

In [40]:
df["has_rating"] = df["review_scores_rating"].notnull().astype(int)
df["review_scores_rating"] = df["review_scores_rating"].fillna(df["review_scores_rating"].median())

In [41]:
def simplify_property_type(x):
    x = x.lower()

    if "boat" in x or "camper" in x:
        return "unique"

    elif any(word in x for word in [
        "condo", "loft", "rental unit",
        "apartment", "serviced apartment", "aparthotel"
    ]):
        return "apartment"

    else:
        return "house"

df["property_group"] = df["property_type"].apply(simplify_property_type)

df["property_group"].value_counts()

property_group
apartment    2424
house         742
unique         10
Name: count, dtype: int64

In [42]:
df = df[df["property_group"] != "unique"].copy()

df["property_group"].value_counts()

property_group
apartment    2424
house         742
Name: count, dtype: int64

In [43]:
print(df.shape)
df.head()
df.isnull().sum()

(3166, 13)


price                      0
accommodates               0
bedrooms                  11
bathrooms                  1
property_type              0
room_type                  0
neighbourhood_cleansed     0
review_scores_rating       0
number_of_reviews          0
availability_365           0
minimum_nights             0
has_rating                 0
property_group             0
dtype: int64

In [44]:
df["bedrooms"] = df["bedrooms"].fillna(df["bedrooms"].median())
df["bathrooms"] = df["bathrooms"].fillna(df["bathrooms"].median())

In [60]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# ===== MODEL A: Emil-variant =====
# Samma idé som Emil: room_type + neighbourhood, men på din rensade data

df_emil = df.copy()

df_emil = pd.get_dummies(
    df_emil,
    columns=["room_type", "neighbourhood_cleansed"],
    drop_first=True
)

features_emil = [
    "accommodates",
    "bedrooms",
    "bathrooms",
    "review_scores_rating",
    "number_of_reviews",
    "availability_365",
    "minimum_nights",
    "has_rating"
] + [
    col for col in df_emil.columns
    if col.startswith("room_type_") or col.startswith("neighbourhood_cleansed_")
]

X_emil = df_emil[features_emil]
y_emil = df_emil["price"]

model_emil = LinearRegression()
model_emil.fit(X_emil, y_emil)

df_emil["predicted_price"] = model_emil.predict(X_emil)
df_emil["deal_score"] = df_emil["predicted_price"] - df_emil["price"]

r2_emil = r2_score(y_emil, df_emil["predicted_price"])
mae_emil = mean_absolute_error(y_emil, df_emil["predicted_price"])

print("=== Emil-variant ===")
print("R²:", round(r2_emil, 3))
print("MAE:", round(mae_emil, 2))
print()

print("Bästa deals enligt Emil-variant:")
print(
    df_emil[["price", "predicted_price", "deal_score"]]
    .sort_values("deal_score", ascending=False)
    .head(10)
)
print()

print("Sämsta deals enligt Emil-variant:")
print(
    df_emil[["price", "predicted_price", "deal_score"]]
    .sort_values("deal_score", ascending=True)
    .head(10)
)
print()


# ===== MODEL B: Din variant =====
# Samma modell, men med property_group också

df_you = df.copy()

df_you = pd.get_dummies(
    df_you,
    columns=["room_type", "neighbourhood_cleansed", "property_group"],
    drop_first=True
)

features_you = [
    "accommodates",
    "bedrooms",
    "bathrooms",
    "review_scores_rating",
    "number_of_reviews",
    "availability_365",
    "minimum_nights",
    "has_rating"
] + [
    col for col in df_you.columns
    if col.startswith("room_type_")
    or col.startswith("neighbourhood_cleansed_")
    or col.startswith("property_group_")
]

X_you = df_you[features_you]
y_you = df_you["price"]

model_you = LinearRegression()
model_you.fit(X_you, y_you)

df_you["predicted_price"] = model_you.predict(X_you)
df_you["deal_score"] = df_you["predicted_price"] - df_you["price"]

r2_you = r2_score(y_you, df_you["predicted_price"])
mae_you = mean_absolute_error(y_you, df_you["predicted_price"])

print("=== Din variant ===")
print("R²:", round(r2_you, 3))
print("MAE:", round(mae_you, 2))
print()

print("Bästa deals enligt din variant:")
print(
    df_you[["price", "predicted_price", "deal_score"]]
    .sort_values("deal_score", ascending=False)
    .head(10)
)
print()

print("Sämsta deals enligt din variant:")
print(
    df_you[["price", "predicted_price", "deal_score"]]
    .sort_values("deal_score", ascending=True)
    .head(10)
)
print()

print("=== Jämförelse ===")
print("R² Emil-variant:", round(r2_emil, 3))
print("R² Din variant:", round(r2_you, 3))
print("MAE Emil-variant:", round(mae_emil, 2))
print("MAE Din variant:", round(mae_you, 2))

=== Emil-variant ===
R²: 0.396
MAE: 572.01

Bästa deals enligt Emil-variant:
       price  predicted_price   deal_score
4442  2743.0      5735.944846  2992.944846
4774  1586.0      4411.577678  2825.577678
4722  1850.0      4227.206482  2377.206482
4229  1658.0      3974.886358  2316.886358
4938  1713.0      3991.803137  2278.803137
2464  1546.0      3750.247251  2204.247251
2249  2546.0      4658.267302  2112.267302
3129  2153.0      4236.210660  2083.210660
4299  1185.0      3247.239753  2062.239753
3072  1350.0      3376.717543  2026.717543

Sämsta deals enligt Emil-variant:
        price  predicted_price    deal_score
2091  17872.0      1803.244580 -16068.755420
3585  16000.0      2059.186484 -13940.813516
2030  15000.0      3057.997863 -11942.002137
114   15000.0      3238.585036 -11761.414964
3285  12000.0      1088.051721 -10911.948279
1511   9459.0      1077.017250  -8381.982750
3568  10000.0      1888.498561  -8111.501439
3620  10800.0      2782.568284  -8017.431716
3574  1000

In [61]:
coeffs = pd.Series(model_you.coef_, index=X_you.columns)
coeffs.sort_values(ascending=False).head(10)

neighbourhood_cleansed_Östermalms      717.021725
neighbourhood_cleansed_Norrmalms       671.951178
neighbourhood_cleansed_Södermalms      655.201648
neighbourhood_cleansed_Kungsholmens    356.022907
bedrooms                               306.686714
review_scores_rating                   270.269540
accommodates                           173.338242
property_group_house                   168.672148
bathrooms                               62.132790
availability_365                         0.758249
dtype: float64